<a href="https://colab.research.google.com/github/Thisis4oF/ComfyUIonColab/blob/main/ComfyUIonColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ปุ่มรันด่วนขั้นตอนทั้งหมดในคลิกเดียว (Master Auto-Launcher)
เชื่อมต่อ Google Drive อัตโนมัติ ติดตั้งแพ็กเกจที่จำเป็น และเปิดเซิร์ฟเวอร์ ComfyUI พร้อมใช้งานทันที

---
**หมายเหตุการแก้ไข:**
- ลบคำสั่ง `drive.flush_and_unmount()` ออก เพื่อป้องกันการตัดการเชื่อมต่อ Drive กะทันหัน
- ลบคำสั่ง `shutil.rmtree` อัตโนมัติ เพื่อป้องกันไม่ให้เผลอลบโฟลเดอร์โมเดล/Workflow บน Google Drive
- ล็อกตำแหน่งโฟลเดอร์ทำงานให้อยู่บน Google Drive เสมอ ไม่สลับไปรันบน `/content/ComfyUI` ชั่วคราว
- เพิ่มการติดตั้ง `gguf` ล่วงหน้าเพื่อป้องกันข้อผิดพลาด Custom Node ComfyUI-GGUF

In [ ]:
#@title 🚀 กดเริ่มทำงานทั้งหมดแบบอัตโนมัติที่นี่ (Google Drive Edition)
import os
import sys
import shutil
import subprocess
import threading
import time
import socket

# --- 1. ตรวจสอบและเชื่อมต่อ Google Drive แบบเสถียร ---
print("🔄 1. กำลังตรวจสอบการเชื่อมต่อ Google Drive...")
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        print("✅ เชื่อมต่อ Google Drive สำเร็จ!")
    except Exception as e:
        print("⚠️ เชื่อมต่อรอบแรกล้มเหลว กำลังลองใหม่:", str(e))
        drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive เชื่อมต่ออยู่แล้ว พร้อมใช้งาน")

# --- 2. ตั้งค่าโฟลเดอร์ทำงานบน Google Drive ---
# ตรวจสอบว่า ComfyUI อยู่ที่ path ใดใน Drive
DRIVE_PATH = "/content/drive/MyDrive/ComfyUI_Drive"
possible_workspaces = [
    "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI",
    "/content/drive/MyDrive/ComfyUI"
]

active_workspace = None
for p in possible_workspaces:
    if os.path.exists(p):
        active_workspace = p
        break

if not active_workspace:
    active_workspace = "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI"
    os.makedirs(DRIVE_PATH, exist_ok=True)
    os.chdir(DRIVE_PATH)
    print("📥 ไม่พบโฟลเดอร์เดิม เริ่มต้นติดตั้ง ComfyUI ลง Google Drive...")
    !git clone https://github.com/comfyanonymous/ComfyUI

print(f"📌 โฟลเดอร์ทำงาน ComfyUI: {active_workspace}")
os.chdir(active_workspace)

# --- 3. ตรวจสอบการอัปเดตซอร์สโค้ด ---
try:
    print("🔄 2. ตรวจสอบอัปเดต ComfyUI ล่าสุด...")
    !git pull
except Exception as e:
    print("⚠️ อัปเดตผ่าน git ไม่สำเร็จ (ใช้งานเวอร์ชันเดิมต่อ):", str(e))

# --- 4. ติดตั้ง Dependencies และ GGUF Package ---
print("🔄 3. กำลังติดตั้งไลบรารี gguf และแพ็กเกจระบบ...")
!pip install -q gguf
if os.path.exists("requirements.txt"):
    !pip install -q xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
else:
    !pip install -q xformers!=0.0.18 torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# ติดตั้ง Cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("🔄 4. กำลังดาวน์โหลดและติดตั้ง Cloudflared Tunnel...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null

# --- 5. เริ่มต้น Tunnel และรัน ComfyUI ---
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n✨ [ระบบ] ComfyUI โหลดสำเร็จแล้ว! กำลังเชื่อมโยงลิ้งก์ภายนอก...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("👉 เข้าใช้งานได้ที่ลิงก์นี้:", l[l.find("http"):].strip())

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🔥 5. กำลังเริ่มสตาร์ทระบบ ComfyUI Server...")
!python main.py --dont-print-server


# ขั้นตอนที่ 1: เชื่อมต่อ Google Drive (Mount Drive)

In [ ]:
#@markdown <center><h3>Mount Google Drive</h3></center>
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive เชื่อมต่อเรียบร้อยแล้ว!")
else:
    print("✅ Google Drive เชื่อมต่ออยู่แล้ว")


In [ ]:
assert __import__('os').path.exists('/content/drive/MyDrive'), 'Google Drive ยังไม่ได้เชื่อมต่อ กรุณารันเซลล์ด้านบนก่อน'
!ls -la /content/drive/MyDrive

# ขั้นตอนที่ 2: ตั้งค่าไดเรกทอรีและ Dependencies

In [ ]:
import os

# กำหนด Path บน Google Drive
DRIVE_PATH = "/content/drive/MyDrive/ComfyUI_Drive"
WORKSPACE = os.path.join(DRIVE_PATH, "ComfyUI")

# หากโฟลเดอร์ ComfyUI อยู่ที่ Drive โดยตรง (เช่น /content/drive/MyDrive/ComfyUI) ให้ตรวจจับอัตโนมัติ
if not os.path.exists(WORKSPACE) and os.path.exists("/content/drive/MyDrive/ComfyUI"):
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    DRIVE_PATH = "/content/drive/MyDrive"

os.makedirs(DRIVE_PATH, exist_ok=True)
%cd {DRIVE_PATH}

if not os.path.exists(WORKSPACE):
    print("-= กำลังติดตั้ง ComfyUI ลงใน Google Drive =-")
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print(f"✅ พบ ComfyUI ที่: {WORKSPACE}")

%cd {WORKSPACE}
!git pull
!pip install -q gguf
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121


# ขั้นตอนที่ 3: จัดการและดาวน์โหลดโมเดล (Models Download)

In [ ]:
#@title 📥 Multi-Category Model Downloader (Civitai & Hugging Face)
import os
import subprocess
from google.colab import userdata

MODEL_TYPE = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings", "unet", "clip"]
DOWNLOAD_URL = "https://civitai.com/api/download/models/2071650" #@param {type:"string"}
CUSTOM_FILENAME = "cyberrealisticPony_v127Alt.safetensors" #@param {type:"string"}

try:
    CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except:
    CIVITAI_API_TOKEN = ""

!apt-get -y install -qq aria2

# ตรวจหา Workspace บน Drive
dest_dir = None
for base in ["/content/drive/MyDrive/ComfyUI_Drive/ComfyUI", "/content/drive/MyDrive/ComfyUI"]:
    if os.path.exists(base):
        dest_dir = f"{base}/models/{MODEL_TYPE}"
        break

if not dest_dir:
    dest_dir = f"/content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/{MODEL_TYPE}"

os.makedirs(dest_dir, exist_ok=True)
print(f"🎯 ปลายทางเก็บโมเดล: {dest_dir}")

final_url = DOWNLOAD_URL
is_civitai = 'civitai.com' in DOWNLOAD_URL

if is_civitai:
    if CIVITAI_API_TOKEN.strip():
        separator = "&" if "?" in DOWNLOAD_URL else "?"
        final_url = f"{DOWNLOAD_URL}{separator}token={CIVITAI_API_TOKEN}"
    else:
        print("⚠️ โมเดลบางตัวบน Civitai ต้องใช้ API Token กรุณาใส่ใน Colab Secrets หากดาวน์โหลดไม่ผ่าน")

out_filename = f"-o \"{CUSTOM_FILENAME}\"" if CUSTOM_FILENAME.strip() else ""
%cd {dest_dir}
print("🚀 กำลังเริ่มดาวน์โหลด...")
result = subprocess.run(f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {out_filename} "{final_url}"', shell=True)

if result.returncode != 0:
    print("\n❌ ดาวน์โหลดด้วย aria2 ไม่สำเร็จ กำลังลองด้วย wget...")
    wget_filename = f"-O \"{CUSTOM_FILENAME}\"" if CUSTOM_FILENAME.strip() else ""
    !wget -c {wget_filename} "{final_url}"
else:
    print("🎉 ดาวน์โหลดโมเดลสำเร็จเรียบร้อยแล้ว!")


## ตรวจสอบไฟล์โมเดลใน Checkpoints

In [ ]:
import os
for base in ["/content/drive/MyDrive/ComfyUI_Drive/ComfyUI", "/content/drive/MyDrive/ComfyUI"]:
    ckpt_path = os.path.join(base, "models/checkpoints")
    if os.path.exists(ckpt_path):
        print(f"📂 รายการโมเดลใน: {ckpt_path}")
        !ls -al "{ckpt_path}"


# ขั้นตอนที่ 4: รันเซิร์ฟเวอร์ ComfyUI ผ่าน Cloudflare Tunnel

In [ ]:
import subprocess
import threading
import time
import socket
import os

# ติดตั้ง gguf และ cloudflared ป้องกันตกหล่น
!pip install -q gguf
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null

# ค้นหาโฟลเดอร์ ComfyUI บน Google Drive โดยตรง (ไม่สลับไปรันบน /content เปล่า)
workspace_path = None
candidate_paths = [
    "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI",
    "/content/drive/MyDrive/ComfyUI"
]

for path in candidate_paths:
    if os.path.exists(path) and os.path.exists(os.path.join(path, "main.py")):
        workspace_path = path
        break

if not workspace_path:
    raise FileNotFoundError("❌ ไม่พบ ComfyUI บน Google Drive กรุณาตรวจสอบว่า Mount Drive สำเร็จและมีโฟลเดอร์ ComfyUI อยู่จริง")

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n[ระบบ] ComfyUI โหลดเสร็จสิ้น กำลังสร้างลิงก์เข้าใช้งานผ่าน Cloudflare Tunnel...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("👉 คลิกที่ลิงก์นี้เพื่อเข้าใช้งาน ComfyUI:", l[l.find("http"):].strip())

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print(f"📌 กำลังเปลี่ยนโฟลเดอร์ทำงานไปที่: {workspace_path}")
os.chdir(workspace_path)
!python main.py --dont-print-server
